In [1]:
from copy import deepcopy
from datetime import date
from pathlib import Path

import numpy as np

from conf.behavior_cloning.diffusion.five_demos.default import config
from rlbench.action_modes.arm_action_modes import BimanualEndEffectorPoseViaIK
from tapas_gmm.encoder.encoder import ObservationEncoderConfig
from tapas_gmm.env.rlbench import RLBenchEnvironment, RLBenchEnvironmentConfig
from tapas_gmm.policy.diffusion import DiffusionPolicy
from tapas_gmm.utils.select_gpu import device

import imageio.v2 as imageio

2026-08-09 12:52:21.896 | INFO     |  Running on cpu


In [2]:
joint_config = deepcopy(config.policy)
leader_config = deepcopy(config.policy)
follower_config = deepcopy(config.policy)

for policy_config in (joint_config, leader_config, follower_config):
    policy_config.obs_dim = 35
    policy_config.horizon = 16
    policy_config.n_obs_steps = 2
    policy_config.n_action_steps = 8
    policy_config.training = None
    policy_config.obs_encoder = ObservationEncoderConfig(
        ee_pose=True,
        object_poses=True,
    )
    policy_config.unet.down_dims = (64, 128, 256)

joint_config.action_dim = 16
joint_config.unet.input_dim = 16
joint_config.unet.global_cond_dim = 35 * 2

leader_config.action_dim = 8
leader_config.arm = "left"
leader_config.unet.input_dim = 8
leader_config.unet.global_cond_dim = 35 * 2

follower_config.action_dim = 8
follower_config.arm = "right"
follower_config.condition_on_arm = "left"
follower_config.unet.input_dim = 8
follower_config.unet.global_cond_dim = 35 * 2 + 16 * 8

In [3]:
checkpoint_root = Path("../../training_runs")
joint_dir = max(checkpoint_root.glob("*_diffusion_joint"))
leader_follower_dir = max(checkpoint_root.glob("*_diffusion_leader_follower"))

joint_policy = DiffusionPolicy(joint_config).to(device)
leader_policy = DiffusionPolicy(leader_config).to(device)
follower_policy = DiffusionPolicy(follower_config).to(device)

joint_policy.from_disk(str(joint_dir / "latest.pt"))
leader_policy.from_disk(str(leader_follower_dir / "leader/latest.pt"))
follower_policy.from_disk(str(leader_follower_dir / "follower/latest.pt"))

joint_policy.eval()
leader_policy.eval()
follower_policy.eval()

2026-08-09 12:52:54.043 | INFO     |  Initializing DiffusionPolicy:
2026-08-09 12:52:54.043 | INFO     |    Initializing Policy:
2026-08-09 12:52:54.814 | INFO     |    number of parameters: 5525968
2026-08-09 12:52:54.909 | INFO     |    No encoder config provided. Using None.
None
2026-08-09 12:52:55.079 | INFO     |    number of parameters: 5522376
None
2026-08-09 12:52:55.250 | INFO     |    number of parameters: 5981128
None


RuntimeError: Error(s) in loading state_dict for DiffusionPolicy:
	size mismatch for model.mid_modules.0.blocks.0.block.0.weight: copying a param with shape torch.Size([1024, 1024, 5]) from checkpoint, the shape in current model is torch.Size([256, 256, 5]).
	size mismatch for model.mid_modules.0.blocks.0.block.0.bias: copying a param with shape torch.Size([1024]) from checkpoint, the shape in current model is torch.Size([256]).
	size mismatch for model.mid_modules.0.blocks.0.block.1.weight: copying a param with shape torch.Size([1024]) from checkpoint, the shape in current model is torch.Size([256]).
	size mismatch for model.mid_modules.0.blocks.0.block.1.bias: copying a param with shape torch.Size([1024]) from checkpoint, the shape in current model is torch.Size([256]).
	size mismatch for model.mid_modules.0.blocks.1.block.0.weight: copying a param with shape torch.Size([1024, 1024, 5]) from checkpoint, the shape in current model is torch.Size([256, 256, 5]).
	size mismatch for model

In [ ]:
tapas_env = RLBenchEnvironment(
    RLBenchEnvironmentConfig(
        action_mode=BimanualEndEffectorPoseViaIK,
        robot_setup="dual_panda",
        task="BimanualDualPushButtons",
        cameras=("front",),
        camera_pose={},
        image_size=(128, 128),
        static=False,
        headless=False,
        scale_action=False,
        delay_gripper=False,
        gripper_plot=False,
        absolute_action_mode=True,
        action_frame="world",
    )
)

In [ ]:
def predict_joint(obs):
    trajectory, _ = joint_policy.predict(obs)
    return np.concatenate((trajectory.ee, trajectory.gripper), axis=-1)


def predict_leader_follower(obs):
    leader_trajectory, leader_info = leader_policy.predict(obs)
    follower_trajectory, _ = follower_policy.predict(
        obs,
        condition=leader_info["action_pred"],
    )
    return np.concatenate(
        (
            leader_trajectory.ee,
            follower_trajectory.ee,
            leader_trajectory.gripper[:, None],
            follower_trajectory.gripper[:, None],
        ),
        axis=-1,
    )

In [ ]:
def run_episode(architecture, max_steps=200):
    obs = tapas_env.reset()
    joint_policy.reset_episode(tapas_env)
    leader_policy.reset_episode(tapas_env)
    follower_policy.reset_episode(tapas_env)

    total_reward = 0
    step = 0
    done = False
    frames = []

    while step < max_steps and not done:
        if architecture == "joint":
            actions = predict_joint(obs)
        else:
            actions = predict_leader_follower(obs)

        for action in actions:
            obs, reward, done, _ = tapas_env.step(action)
            total_reward += reward
            step += 1

            if obs is not None:
                frame = obs.cameras["front"].rgb

                if frame.ndim == 4:
                    frame = frame[0]

                if frame.shape[0] == 3:
                    frame = frame.permute(1, 2, 0)

                frame = frame.clamp(0, 1).mul(255).byte().cpu().numpy()
                frames.append(frame)

            if done or obs is None or step >= max_steps:
                break

    video_dir = Path("../artifacts/videos/diffusion") / architecture / date.today().isoformat()
    video_dir.mkdir(parents=True, exist_ok=True)
    imageio.mimsave(video_dir / "run.mp4", frames, fps=20)

    print("architecture:", architecture)
    print("steps:", step)
    print("total_reward:", total_reward)

In [ ]:
run_episode("joint")

In [ ]:
run_episode("leader_follower")

In [ ]:
tapas_env.close()